In [26]:
import json
import os
import google.generativeai as genai
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from typing import Optional, List, Dict, Any
import time
from typing import Optional

In [22]:
# Load API key
load_dotenv(dotenv_path=r"C:\Users\NT\ai-final-project\.env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel("gemini-2.0-flash-lite")

# Load chunks and index from Level 1
with open(r"C:\Users\NT\ai-final-project\chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

embeddings = np.load(r"C:\Users\NT\ai-final-project\embeddings.npy")
DIM = embeddings.shape[1]

index = faiss.read_index(r"C:\Users\NT\ai-final-project\recipe_index.faiss")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Setup done!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Setup done!


In [23]:
# Query Rewriting
def rewrite_query(query: str) -> str:
    prompt = f"""You are a search query optimizer for a recipe assistant.
Rewrite the following user query to make it cleaner, more specific, and easier to search.
Return ONLY the rewritten query, nothing else.

Original query: {query}

Rewritten query:"""
    
    try:
        response = gemini.generate_content(prompt)
        rewritten = response.text.strip()
        return rewritten
    except Exception as e:
        return query  

# Test
original = "pancakes?"
rewritten = rewrite_query(original)
print(f"Original : {original}")
print(f"Rewritten: {rewritten}")

Original : pancakes?
Rewritten: pancakes?


In [ ]:
def classify_query(query: str) -> str:
    """
    Classify the query into: factual, recommendation, general, or out_of_scope.
    """
    if not GEMINI_API_KEY:
        return "general"
    
    prompt = f"""You are a strict query classifier. Your task is to classify the user's question into EXACTLY ONE category.

Categories (choose ONLY ONE):
- factual: asking for specific facts (ingredients, steps, time, temperature, measurements)
- recommendation: asking for suggestions, ideas, or what to cook
- general: general cooking/food question
- out_of_scope: not related to food or cooking

CRITICAL RULE: Return ONLY the category name as a single word. Do NOT add any explanation, punctuation, or extra text.

Examples:
User: "How do I make pancakes?" → factual
User: "What should I cook for dinner?" → recommendation  
User: "How long do I bake a cake?" → factual
User: "What is the capital of France?" → out_of_scope
User: "I have eggs and bananas, what can I make?" → recommendation
User: "Is this recipe healthy?" → general

Now classify this query. Return ONLY the category name, nothing else:
User: "{query}"
→"""

    try:
        response = gemini.generate_content(prompt)
        category = response.text.strip().lower()

        valid_categories = ["factual", "recommendation", "general", "out_of_scope"]

        for cat in valid_categories:
            if cat in category:
                return cat
  
        first_word = category.split()[0] if category.split() else ""
        if first_word in valid_categories:
            return first_word

        print(f" Unknown response from Gemini: '{category}'")
        return "general"
        
    except Exception as e:
        print(f" Classification error: {e}")
        return "general"

In [25]:
# Test classification
test_queries = [
    "How do I make pancakes?"
]

for q in test_queries:
    result = classify_query(q)
    print(f"Query   : {q}")
    print(f"Class   : {result}")
    print()

⚠️ Classification error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite
Please retry in 42.255468367s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPer

In [21]:
# Filter Extraction
def extract_filters(query: str) -> dict:
    prompt = f"""You are a filter extractor for a recipe assistant.
Extract filters from the following query and return them as JSON.
Possible filters:
- category: one of [Breakfast, Dinner, Desserts, Salads, Soups] or null
- difficulty: one of [easy, medium, hard] or null
- main_ingredient: main ingredient mentioned or null

Return ONLY a valid JSON object, nothing else.

Query: {query}

JSON:"""
    
    try:
        response = gemini.generate_content(prompt)
        text = response.text.strip()
        text = text.replace("```json", "").replace("```", "").strip()
        filters = json.loads(text)
        return filters
    except Exception as e:
        return {"category": None, "difficulty": None, "main_ingredient": None}

# Test
queries = [
    "easy breakfast pancake recipe",
    "simple chicken soup for dinner",
    "chocolate dessert with eggs",
]

for q in queries:
    print(f"Query  : {q}")
    print(f"Filters: {extract_filters(q)}")
    print()

Query  : easy breakfast pancake recipe
Filters: {'category': None, 'difficulty': None, 'main_ingredient': None}

Query  : simple chicken soup for dinner
Filters: {'category': None, 'difficulty': None, 'main_ingredient': None}

Query  : chocolate dessert with eggs
Filters: {'category': None, 'difficulty': None, 'main_ingredient': None}



In [27]:
def retrieve(query: str, top_k: int = 5, category_filter: Optional[str] = None) -> list[dict]:
    """
    Retrieve the top_k most similar chunks. If a category_filter is provided,
    it will first filter the chunks by that category before performing the search.
    """
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)
    
    if category_filter:
        filtered_indices = []
        for i, chunk in enumerate(chunks):
            if chunk["metadata"]["category"].lower() == category_filter.lower():
                filtered_indices.append(i)
        
        if not filtered_indices:
            print(f"No chunks found for category: {category_filter}")
            return []
   
        filtered_embeddings = embeddings[filtered_indices]
        sub_index = faiss.IndexFlatL2(DIM)
        sub_index.add(filtered_embeddings.astype(np.float32))
        
        distances, sub_indices = sub_index.search(query_vec, min(top_k, len(filtered_indices)))
      
        results = []
        for dist, sub_idx in zip(distances[0], sub_indices[0]):
            if sub_idx == -1:
                continue
            original_idx = filtered_indices[sub_idx]
            chunk = chunks[original_idx]
            results.append({
                "text": chunk["text"],
                "metadata": chunk["metadata"],
                "similarity_score": float(dist),
            })
        return results

    distances, indices = index.search(query_vec, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        chunk = chunks[idx]
        results.append({
            "text": chunk["text"],
            "metadata": chunk["metadata"],
            "similarity_score": float(dist),
        })
    return results

In [28]:
def build_prompt(query: str, retrieved_chunks: list) -> str:
    """
    Build prompt with retrieved context for Gemini.
    """
    context = ""
    for i, chunk in enumerate(retrieved_chunks, 1):
        meta = chunk["metadata"]
        context += f"""
--- Source {i} ---
Title: {meta.get('title', '')}
Category: {meta.get('category', '')}
{chunk['text']}

"""
    
    prompt = f"""
You are a recipe assistant. Answer the question using ONLY the information from the recipes provided below.

Rules:
- ONLY use information from the provided recipes
- Do NOT invent or add information
- If the answer is not in the recipes, say "I don't have enough information"

Context:
{context}

Question: {query}

Answer:"""
    return prompt

In [29]:
def answer_question(query: str, top_k: int = 5) -> dict:
    """
    Full RAG pipeline with Level 2 Query Intelligence.
    """
    print("\n" + "="*50)
    print(f"Original Query: {query}")
    print("="*50)
    
    rewritten = rewrite_query(query)
    print(f"\n Rewritten Query: {rewritten}")
    
    category = classify_query(query)
    print(f"\n Query Class: {category}")
    
    filters = extract_filters(query)
    print(f"\n Extracted Filters: {json.dumps(filters, indent=2)}")
    
    category_filter = filters.get("category")
    retrieved = retrieve(rewritten, top_k, category_filter)
    
    if not retrieved:
        print("\n No relevant recipes found.")
        return {
            "query": query,
            "rewritten": rewritten,
            "class": category,
            "filters": filters,
            "answer": "No relevant recipes found.",
            "sources": []
        }
    
    print(f"\n Retrieved Sources ({len(retrieved)} chunks):")
    for i, r in enumerate(retrieved, 1):
        print(f"  {i}. {r['metadata']['title']} (Score: {r['similarity_score']:.3f})")
    
    #Generate Answer
    prompt = build_prompt(rewritten, retrieved)
    try:
        response = gemini.generate_content(prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {e}"
    
    print(f"\n Final Answer:\n{answer}")
    
    return {
        "query": query,
        "rewritten": rewritten,
        "class": category,
        "filters": filters,
        "answer": answer,
        "sources": retrieved
    }

In [ ]:
test_queries = [
    "How do I make pancakes for breakfast?",
    "Easy salad with tomatoes and cucumbers",
    "What should I cook for dinner tonight?"
]

for q in test_queries:
    result = answer_question(q, top_k=3)
    print("\n" + "="*60 + "\n")
    time.sleep(10)



Original Query: How do I make pancakes for breakfast?

 Rewritten Query: How do I make pancakes for breakfast?
⚠️ Classification error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite
Please retry in 22.115872024s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generat